# 🎬 OpenSource Clipping — T4 Quick Start

Minimal configuration for Google Colab T4 GPU.

**How to use:**
1. Run **Cell 1 (Setup)** once.
2. Fill in **Cell 2 (Configuration)** — defaults are already T4-optimized.
3. Run **Cell 3** to validate.
4. Run **Cell 4** to generate clips.

All other options are pre-tuned for T4 GPU.

## ① Setup — Install dependencies

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ═══════════════════════════════════════════════════════════════
import os, subprocess, sys, shutil, re
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

REPO_URL = "https://github.com/troublescope/Ashortai.git"
CLONE_DIR = "/content/opensource-clipping"
Path(CLONE_DIR).mkdir(parents=True, exist_ok=True)

if os.path.isdir(f"{CLONE_DIR}/.git"):
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)

os.chdir(CLONE_DIR)

print("⏳ Installing dependencies…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# ── Backward-compat patch: GOOGLE_API_KEY only required for Gemini provider ──
main_py = Path(CLONE_DIR) / "main.py"
main_text = main_py.read_text(encoding="utf-8")
old_check = '    if not cfg.api_key_gemini:'
new_check = '    if cfg.ai_provider == "gemini" and not cfg.api_key_gemini:'
if old_check in main_text and new_check not in main_text:
    main_py.write_text(main_text.replace(old_check, new_check), encoding="utf-8")
    print("🩹 Patched main.py: GOOGLE_API_KEY only required when provider is gemini.")
print("✅ Dependencies installed.")

try:
    DRIVE_CACHE = Path("/content/drive/MyDrive/osc_cache")
    DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
    for m in ["blaze_face_full_range.tflite", "face_yolov8m.pt", "face_yolov8n.pt", "face_yolov8s.pt"]:
        src, dst = DRIVE_CACHE / m, Path(CLONE_DIR) / m
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            print(f"📥 Restored {m}")
except Exception:
    pass

print("\n✅ Setup complete.")

## ⚙️ Configuration

All user-configurable settings are in the cell below. T4-optimized defaults are pre-selected. Only change what you need; leave the rest as-is.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — ALL CONFIGURATION
# ═══════════════════════════════════════════════════════════════

# ── 📥 INPUT ──
VIDEO_URL = "" #@param {type:"string"}
SOURCE_PLATFORM = "youtube" #@param ["youtube", "tiktok", "instagram", "gdrive"]

# ── 🤖 AI SETTINGS ──
AI_PROVIDER = "ollama" #@param ["gemini", "nvidia", "ollama"]
AI_MODEL = "gemini-3-flash-preview:cloud" #@param {type:"string"}
OLLAMA_URL = "https://ollama.com" #@param {type:"string"}
OLLAMA_FALLBACK_URL = "https://ollama.com" #@param {type:"string"}
OLLAMA_FALLBACK_MODEL = "gemini-3-flash-preview:cloud" #@param {type:"string"}
GEMINI_MODEL = "gemini-3-flash-preview" #@param {type:"string"}
OLLAMA_API_KEY = "" #@param {type:"string"}
GOOGLE_API_KEY = "" #@param {type:"string"}
NVIDIA_API_KEY = "" #@param {type:"string"}

# ── ✂️ CLIPPING SETTINGS ──
MAX_CLIPS = 5 #@param {type:"integer"}
ASPECT_RATIO = "9:16" #@param ["9:16", "16:9", "1:1", "3:4", "4:5"]
RENDER_HEIGHT = "1080" #@param ["source", "1080", "1440"]
SOURCE_HEIGHT = "max" #@param ["max", "1080", "1440", "2160"]
FONT_STYLE = "HORMOZI" #@param ["DEFAULT", "HORMOZI", "STORYTELLER", "CINEMATIC"]
FACE_DETECTOR = "mediapipe" #@param ["mediapipe", "yolo"]
YOLO_SIZE = "8m" #@param ["8n", "8s", "8m", "8n_v2", "9c"]

# ── 💬 SUBTITLE & TRANSCRIPTION ──
ENABLE_SUBTITLES = True #@param {type:"boolean"}
USE_KARAOKE_EFFECT = True #@param {type:"boolean"}
WORDS_PER_SUBTITLE = 5 #@param {type:"integer"}
USE_YT_TRANSCRIPT_API = True #@param {type:"boolean"}
USE_DLP_SUBS = True #@param {type:"boolean"}
WHISPER_MODEL = "large-v3" #@param {type:"string"}
WHISPER_DEVICE = "auto" #@param ["auto", "cuda", "cpu"]
WHISPER_COMPUTE_TYPE = "float16" #@param ["float16", "float32", "int8"]

# ── 📦 OUTPUT & QUALITY ──
VIDEO_PRESET = "ultrafast" #@param ["ultrafast", "fast", "slow", "veryslow", "auto"]
VIDEO_SCALE_ALGO = "bicubic" #@param ["lanczos", "bicubic", "bilinear", "area"]
VIDEO_CQ = 23 #@param {type:"integer"}
VIDEO_CRF = 20 #@param {type:"integer"}
VIDEO_SHARPEN = False #@param {type:"boolean"}
ENABLE_COLAB_CLEANUP = True #@param {type:"boolean"}

# ── 🔧 ADVANCED / OPTIONAL ──
DISABLE_BGM = True #@param {type:"boolean"}
DISABLE_BROLL = True #@param {type:"boolean"}
DISABLE_HOOK_GLITCH = False #@param {type:"boolean"}
ENABLE_HOOK_V2 = False #@param {type:"boolean"}
HOOK_DURATION = 3 #@param {type:"integer"}
ENABLE_SPLIT_SCREEN = False #@param {type:"boolean"}
SPLIT_TRIGGER = "diarization" #@param ["diarization", "face"]
ENABLE_CAMERA_SWITCH = False #@param {type:"boolean"}
DISABLE_SEGMENT_TRIM = False #@param {type:"boolean"}
AGGRESSIVE_SILENCE_TRIM = False #@param {type:"boolean"}
HF_TOKEN = "" #@param {type:"string"}

## ✅ Validate

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — VALIDATION
# ═══════════════════════════════════════════════════════════════
errors = []
warnings = []

if not VIDEO_URL.strip():
    errors.append("❌ VIDEO_URL is required.")
elif not VIDEO_URL.strip().startswith(("http://", "https://")):
    errors.append("❌ VIDEO_URL must start with http:// or https://.")

if AI_PROVIDER == "ollama" and OLLAMA_URL.startswith("https://ollama.com") and not OLLAMA_API_KEY.strip():
    errors.append("❌ OLLAMA_API_KEY is required for Ollama Cloud.")
if AI_PROVIDER == "gemini" and not GOOGLE_API_KEY.strip():
    errors.append("❌ GOOGLE_API_KEY is required when AI_PROVIDER is 'gemini'.")
if AI_PROVIDER == "nvidia" and not NVIDIA_API_KEY.strip():
    errors.append("❌ NVIDIA_API_KEY is required when AI_PROVIDER is 'nvidia'.")

if AI_PROVIDER != "gemini" and not GOOGLE_API_KEY.strip():
    warnings.append("⚠️ GOOGLE_API_KEY is empty. The pipeline cannot fallback to Gemini if Ollama/NVIDIA fails.")

if not (1 <= MAX_CLIPS <= 10):
    errors.append("❌ MAX_CLIPS must be between 1 and 10.")
if not (1 <= WORDS_PER_SUBTITLE <= 20):
    errors.append("❌ WORDS_PER_SUBTITLE must be between 1 and 20.")
if not (1 <= HOOK_DURATION <= 10):
    errors.append("❌ HOOK_DURATION must be between 1 and 10 seconds.")
if not (15 <= VIDEO_CQ <= 50):
    errors.append("❌ VIDEO_CQ must be between 15 and 50.")
if not (15 <= VIDEO_CRF <= 50):
    errors.append("❌ VIDEO_CRF must be between 15 and 50.")

if ASPECT_RATIO != "9:16" and (ENABLE_SPLIT_SCREEN or ENABLE_CAMERA_SWITCH):
    warnings.append("⚠️ Split-screen and camera-switch work best with 9:16 aspect ratio.")
if (ENABLE_SPLIT_SCREEN or ENABLE_CAMERA_SWITCH) and not HF_TOKEN.strip():
    errors.append("❌ HF_TOKEN is required for split-screen or camera-switch mode.")

print("═" * 60)
print("📋 SETTINGS")
print("═" * 60)
print(f"  Video URL           : {VIDEO_URL or '—'}")
print(f"  Source Platform     : {SOURCE_PLATFORM}")
print(f"  AI Provider         : {AI_PROVIDER}")
print(f"  AI Model            : {AI_MODEL}")
print(f"  Ollama API Key      : {'✅ set' if OLLAMA_API_KEY.strip() else '—'}")
print(f"  Google API Key      : {'✅ set' if GOOGLE_API_KEY.strip() else '—'}")
print(f"  NVIDIA API Key      : {'✅ set' if NVIDIA_API_KEY.strip() else '—'}")
print(f"  Max Clips           : {MAX_CLIPS}")
print(f"  Aspect Ratio        : {ASPECT_RATIO}")
print(f"  Render Height       : {RENDER_HEIGHT}")
print(f"  Subtitles           : {ENABLE_SUBTITLES}")
print(f"  YouTube Transcript  : {USE_YT_TRANSCRIPT_API}")
print(f"  Colab Cleanup       : {ENABLE_COLAB_CLEANUP}")

if warnings:
    print("\n⚠️ WARNINGS")
    for w in warnings:
        print(f"  {w}")

if errors:
    print("\n❌ ERRORS")
    for e in errors:
        print(f"  {e}")
    raise SystemExit("Validation failed.")

print("\n✅ Ready to run!")

## 🚀 Run Pipeline

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — RUN PIPELINE
# ═══════════════════════════════════════════════════════════════
import subprocess, sys, shutil, time
from datetime import datetime
from pathlib import Path

# ── Write .env ──
env_lines = []
if OLLAMA_API_KEY.strip(): env_lines.append(f"OLLAMA_API_KEY={OLLAMA_API_KEY.strip()}")
if GOOGLE_API_KEY.strip(): env_lines.append(f"GOOGLE_API_KEY={GOOGLE_API_KEY.strip()}")
if NVIDIA_API_KEY.strip(): env_lines.append(f"NVIDIA_API_KEY={NVIDIA_API_KEY.strip()}")
if HF_TOKEN.strip(): env_lines.append(f"HF_TOKEN={HF_TOKEN.strip()}")
if env_lines:
    Path(CLONE_DIR, ".env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")
    print("🔐 .env written.")

url = VIDEO_URL.strip()

# ── Unique output folder ──
def _slugify(s: str) -> str:
    s = s.split("?")[0].split("/")[-1]
    s = re.sub(r"[^\w\-_. ]", "", s).strip()
    return s[:40] or "clip"

output_slug = f"{_slugify(url)}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = Path(CLONE_DIR) / "outputs" / output_slug
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Output target: {OUTPUT_DIR}")

cmd = [
    sys.executable, "main.py",
    "--url", url,
    "--source", SOURCE_PLATFORM,
    "--clips", str(MAX_CLIPS),
    "--ratio", ASPECT_RATIO,
    "--render-height", str(RENDER_HEIGHT),
    "--source-height", str(SOURCE_HEIGHT),
    "--font-style", FONT_STYLE,
    "--face-detector", FACE_DETECTOR,
    "--yolo-size", YOLO_SIZE,
    "--ai-provider", AI_PROVIDER,
    "--gemini-model", GEMINI_MODEL,
    "--ollama-url", OLLAMA_URL,
    "--ollama-model", AI_MODEL,
    "--ollama-fallback-url", OLLAMA_FALLBACK_URL,
    "--ollama-fallback-model", OLLAMA_FALLBACK_MODEL,
    "--whisper-model", WHISPER_MODEL,
    "--whisper-device", WHISPER_DEVICE,
    "--whisper-compute-type", WHISPER_COMPUTE_TYPE,
    "--video-preset", VIDEO_PRESET,
    "--video-scale-algo", VIDEO_SCALE_ALGO,
    "--video-cq", str(VIDEO_CQ),
    "--video-crf", str(VIDEO_CRF),
    "--words-per-sub", str(WORDS_PER_SUBTITLE),
    "--hook-duration", str(HOOK_DURATION),
]

def add_flag(flag, cond):
    if cond:
        cmd.append(flag)

add_flag("--colab-cleanup", ENABLE_COLAB_CLEANUP)
add_flag("--use-yt-transcript-api", USE_YT_TRANSCRIPT_API)
add_flag("--use-dlp-subs", USE_DLP_SUBS)
add_flag("--no-bgm", DISABLE_BGM)
add_flag("--no-broll", DISABLE_BROLL)
add_flag("--no-subs", not ENABLE_SUBTITLES)
add_flag("--no-karaoke", not USE_KARAOKE_EFFECT)
add_flag("--no-hook", DISABLE_HOOK_GLITCH)
add_flag("--camera-switch", ENABLE_CAMERA_SWITCH)
add_flag("--hook-v2", ENABLE_HOOK_V2)
add_flag("--no-segment-trim", DISABLE_SEGMENT_TRIM)
add_flag("--silence-trim", AGGRESSIVE_SILENCE_TRIM)
add_flag("--video-sharpen", VIDEO_SHARPEN)
add_flag("--split-screen", ENABLE_SPLIT_SCREEN)

if ENABLE_SPLIT_SCREEN:
    cmd += ["--split-trigger", SPLIT_TRIGGER]

print("🚀 Starting pipeline…")
print(f"URL: {url}")
print("Command:", " ".join(cmd))
print("─" * 60)

start = time.time()
result = subprocess.run(cmd, cwd=CLONE_DIR)
if result.returncode != 0:
    print(f"❌ Pipeline exited with code {result.returncode}")
elapsed = time.time() - start

print("─" * 60)
print(f"⏱️ Total time: {elapsed/60:.1f} minutes")

# ── Collect outputs into unique folder ──
base_out = Path(CLONE_DIR) / "outputs"
if base_out.exists():
    moved = 0
    for f in sorted(base_out.iterdir()):
        if f.is_file():
            try:
                shutil.move(str(f), str(OUTPUT_DIR / f.name))
                moved += 1
            except Exception as e:
                print(f"⚠️ Could not move {f.name}: {e}")
    print(f"\n📁 {moved} file(s) moved to: {OUTPUT_DIR}")

    files = sorted(OUTPUT_DIR.glob("*_ready.mp4")) + sorted(OUTPUT_DIR.glob("*.jpg"))
    if files:
        print(f"\n📦 Output files ({len(files)}):")
        for f in files:
            print(f"   • {f.name} ({f.stat().st_size/1024/1024:.1f} MB)")
    if list(OUTPUT_DIR.iterdir()):
        zip_path = OUTPUT_DIR / "outputs.zip"
        shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(OUTPUT_DIR))
        print(f"\n🗜️ outputs.zip → {zip_path.stat().st_size/1024/1024:.1f} MB")
        try:
            from google.colab import files
            files.download(str(zip_path))
        except Exception:
            pass
else:
    print("⚠️ outputs/ directory not found.")